# NB_R03 · 국면 스냅샷 게시 (API · 웹 대시보드 · PostgreSQL 전달용)

**[시장국면분석ML] 3/3 단계.** 국면 구간·요약·전이·유사 시기를 계산하고, 웹 대시보드와 API가 그대로 쓰는 **스냅샷 JSON 1개**를 만듭니다. 검사를 통과해야만 `latest`가 바뀝니다. 실패하면 이전 정상 스냅샷이 유지됩니다.

- 출력 파일: `Files/regime/snapshots/<버전>/regime_snapshot.json`, `certificate.json`, `Files/regime/latest.json`, `Files/regime/health.json`
- 출력 Delta: `gold_regime_runs`, `gold_regime_summary`, `gold_regime_transition`, `gold_regime_similar`
- PostgreSQL 복사용 단계 테이블: `regime_pg_snapshot`(1행), `regime_pg_daily` → 파이프라인 Copy 작업이 `regime.snapshot`, `regime.daily`로 적재

In [ ]:
RUN_ID = ""
DATA_MODE = "fabric_gold"   # 운영은 항상 fabric_gold. 합성 데이터는 게시 검사에서 차단됩니다

In [ ]:
# 모듈 무결성 확인 후 불러오기 (팀 Gold 노트북과 같은 해시 확인 방식)
import sys, json, hashlib, importlib, time
from pathlib import Path
import numpy as np, pandas as pd
assert Path("/lakehouse/default/Files").is_dir(), "기본 Lakehouse LH_MarketRegime을 연결하세요."
# 팀 원천 Lakehouse(lh_whatif)에는 쓰지 않습니다. 기본 Lakehouse가 lh_whatif이면 여기서 멈춥니다.
try:
    import notebookutils
    _lh = (notebookutils.runtime.context.get("defaultLakehouseName") or "")
except Exception:
    _lh = ""
if _lh.lower() == "lh_whatif" or Path("/lakehouse/default/Files/bronze").exists():
    raise RuntimeError("TEAM_LAKEHOUSE_DETECTED: 기본 Lakehouse를 LH_MarketRegime으로 연결하세요.")
MODULE_DIR = Path("/lakehouse/default/Files/reference/market_regime/v1")
_manifest = json.loads((MODULE_DIR / "manifest.json").read_text(encoding="utf8"))
for _f, _h in _manifest["files"].items():
    if hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest() != _h:
        raise RuntimeError("REGIME_MODULE_HASH_MISMATCH:" + _f)
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import hmm_numpy, regime_core as rc, fabric_io as fio
for _m in (hmm_numpy, rc, fio):
    importlib.reload(_m)
try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")   # 팀 Gold 노트북과 같은 기준
except Exception:
    pass
print("market_regime", rc.CORE_VERSION, rc.MODEL_VERSION)

In [ ]:
RUN_ID = fio.resolve_run_id(RUN_ID)
rd = fio.run_dir(RUN_ID)
ctx, model_info = fio.read_json(rd / "context.json"), fio.read_json(rd / "model.json")
bic, validation = fio.read_json(rd / "bic.json"), fio.read_json(rd / "validation.json")
if not all([ctx, model_info, bic, validation]):
    raise RuntimeError("RUN_ARTIFACTS_MISSING: NB_R01, NB_R02를 같은 RUN_ID로 먼저 실행하세요")
cfg = ctx["config"]
feat = fio.load_table(spark, "gold_regime_features", "trading_date")
num = [c for c in feat.columns if c not in ("date_ok",)]
feat[num] = feat[num].apply(pd.to_numeric, errors="coerce")
daily = fio.load_table(spark, "gold_regime_daily", "trading_date")
for c in ["p_bull", "p_sideways", "p_bear", "kospi_close"]:
    daily[c] = pd.to_numeric(daily[c], errors="coerce")
daily["run_id"] = daily["run_id"].astype(int); daily["run_day"] = daily["run_day"].astype(int)
print("RUN_ID", RUN_ID, "| daily", daily.shape, daily.index.max().date())

In [ ]:
runs = rc.build_runs(daily, feat)
summary = rc.regime_summary(daily, feat, runs)
trans = rc.transition_tables(daily, runs)
sim, sim_query = rc.similar_periods(feat, daily, None, cfg)
quality = ctx["quality"]; quality["warnings"] = rc.quality_warnings(bic, quality)
snap = rc.build_snapshot(feat, daily, runs, summary, trans, sim, sim_query, validation, model_info, bic,
                         quality, cfg, DATA_MODE, ctx["gold_version"])
payload, sha = rc.snapshot_bytes(snap)
fails = rc.publish_checks(snap, ctx["gold_version"])
if len(payload) > 5_000_000:
    fails.append("PAYLOAD_TOO_LARGE")
print("snapshot", snap["snapshot_version"], "| as_of", snap["as_of"], "|", len(payload), "bytes | sha", sha[:12])
print("현재:", json.dumps({k: snap["current"][k] for k in ["regime_name", "confidence", "days_in_regime", "regime_since", "pending_switch"]}, ensure_ascii=False))
if fails:
    fio.write_json_atomic(fio.REGIME_ROOT / "health.json", {"status": "blocked", "checked_at_utc": fio.now_utc().isoformat(),
                                                            "run_id": RUN_ID, "reasons": fails})
    raise RuntimeError("REGIME_PUBLISH_BLOCKED: " + ", ".join(fails))

In [ ]:
# 1) 스냅샷 파일 + 인증서 (먼저 기록하고 해시 대조)
ver = snap["snapshot_version"]
sdir = fio.REGIME_ROOT / "snapshots" / ver
sdir.mkdir(parents=True, exist_ok=False)
(sdir / "regime_snapshot.json").write_bytes(payload)
if fio.sha256_file(sdir / "regime_snapshot.json") != sha:
    raise RuntimeError("SNAPSHOT_WRITE_HASH_MISMATCH")
cert = {"snapshot_version": ver, "run_id": RUN_ID, "as_of": snap["as_of"], "sha256": sha, "bytes": len(payload),
        "data_mode": DATA_MODE, "model_version": model_info["model_version"], "core_version": rc.CORE_VERSION,
        "gold_version": ctx["gold_version"], "module_manifest": _manifest, "created_at_utc": fio.now_utc().isoformat()}
fio.write_json_atomic(sdir / "certificate.json", cert)

# 2) 분석 Delta 테이블
fio.save_table(spark, runs, "gold_regime_runs")
fio.save_table(spark, summary, "gold_regime_summary")
fio.save_table(spark, trans[0], "gold_regime_transition")
fio.save_table(spark, sim.assign(snapshot_version=ver), "gold_regime_similar")

# 보고서(Power BI)용 현재 상태 1행 + 검증 지표
cur = snap["current"]
_ps = cur.get("pending_switch") or {}
current_tbl = pd.DataFrame([{"as_of": pd.Timestamp(snap["as_of"]), "regime_code": cur["regime_code"],
                             "regime_name": cur["regime_name"], "confidence": float(cur["confidence"] or 0),
                             "p_bull": float(cur["probabilities"]["bull"]), "p_sideways": float(cur["probabilities"]["sideways"]),
                             "p_bear": float(cur["probabilities"]["bear"]), "raw_argmax_code": str(cur.get("raw_argmax_code") or ""),
                             "days_in_regime": int(cur["days_in_regime"]), "regime_since": pd.Timestamp(cur["regime_since"]),
                             "changes_last_252d": int(cur.get("changes_last_252d") or 0),
                             "pending_to_name": str(_ps.get("to_name") or ""), "pending_days": int(_ps.get("consecutive_days") or 0),
                             "pending_required_days": int(_ps.get("required_days") or 0),
                             "snapshot_version": ver, "notice": snap["notice"]}])
fio.save_table(spark, current_tbl, "gold_regime_current")
_v, _rows = validation, []
for _e in _v["events"]:
    _rows += [{"grp": "event", "item": _e["name"], "metric": "bear_share", "value": float(_e["bear_share"] or 0)},
              {"grp": "event", "item": _e["name"], "metric": "mean_p_bear", "value": float(_e["mean_p_bear"] or 0)}]
_rows += [{"grp": "stability", "item": "label", "metric": "switches_per_year", "value": float(_v["stability"]["label_switches_per_year"] or 0)},
          {"grp": "agreement", "item": "filtered_vs_smoothed", "metric": "agreement", "value": float(_v["filtered_vs_smoothed_agreement"] or 0)},
          {"grp": "agreement", "item": "knn", "metric": "agreement", "value": float(_v["knn_regime_agreement"]["mean"] or 0)},
          {"grp": "agreement", "item": "knn", "metric": "random_baseline", "value": float(_v["knn_regime_agreement"]["random_baseline"] or 0)},
          {"grp": "silhouette", "item": "hmm", "metric": "silhouette", "value": float(_v["silhouette"]["hmm"] or 0)},
          {"grp": "silhouette", "item": "gmm", "metric": "silhouette", "value": float(_v["silhouette"]["gmm"] or 0)},
          {"grp": "coverage", "item": "out_of_sample", "metric": "days", "value": float(_v["oos_days"])}]
fio.save_table(spark, pd.DataFrame(_rows).assign(oos_start=str(_v["oos_start"]), snapshot_version=ver), "gold_regime_validation")

# 3) PostgreSQL 복사용 단계 테이블 (이번 실행분만)
created = pd.Timestamp(fio.now_utc()).tz_convert("UTC").tz_localize(None)
pg_snapshot = pd.DataFrame([{"snapshot_version": ver, "as_of": pd.Timestamp(snap["as_of"]),
                             "model_version": model_info["model_version"], "data_mode": DATA_MODE,
                             "gold_snapshot_version": str(ctx["gold_version"].get("gold_snapshot_version") or ""),
                             "payload_text": payload.decode("utf-8"), "payload_sha256": sha, "created_at": created}])
pg_daily = daily.reset_index()[["trading_date", "kospi_close", "regime_code", "p_bull", "p_sideways", "p_bear",
                                "raw_code", "label_basis"]].assign(snapshot_version=ver)
fio.save_table(spark, pg_snapshot, "regime_pg_snapshot")
fio.save_table(spark, pg_daily, "regime_pg_daily")

# 4) 최신 포인터와 상태 (모든 기록이 끝난 뒤 교체)
fio.write_json_atomic(fio.REGIME_ROOT / "latest.json", {"version": ver, "path": f"regime/snapshots/{ver}/regime_snapshot.json",
                                                        "sha256": sha, "as_of": snap["as_of"]})
fio.write_json_atomic(fio.REGIME_ROOT / "health.json", {"status": "ready", "checked_at_utc": fio.now_utc().isoformat(),
                                                        "run_id": RUN_ID, "snapshot_version": ver, "as_of": snap["as_of"],
                                                        "regime": snap["current"]["regime_code"]})
print("게시 완료:", ver, "| latest.json 갱신 | PostgreSQL 단계 테이블 준비")